In [1]:
%run Coding_Crypto_Lib.ipynb

In [2]:
def compute_C_i_j(G_minus, G):
    G_square = square_code_generator_full_rank(G)
    C_square = LinearCode(G_square)
    H_square = C_square.parity_check_matrix()
    V_plus = G.row_space()
    for g in G_minus.rows():
        D_g = diagonal_matrix(g)
        H_g = H_square * D_g
        # try to get rid of this check
        C_g = LinearCode(H_g)
        V_g = C_g.parity_check_matrix().row_space()
        V_plus = V_plus.intersection(V_g)
    return matrix(V_plus.basis())

def compute_all_C_i_j(G):
    G_rref = G.rref()
    G_rref = G_rref[:G_rref.rank()]
    k = G_rref.nrows()
    C_i_j_dict = {}
    C_i_j_dict[(0, 0)] = G_rref
    C_i_j_dict[(1, 0)] = G_rref[1:]
    C_0_1_list = G_rref.rows()
    C_0_1_list = [C_0_1_list[0]] + C_0_1_list[2:]
    C_i_j_dict[(0, 1)] = matrix(C_0_1_list)
    C_i_j_dict[(1, 1)] = G_rref[2:]
    for i in range(1, k-1):
        C_i_j_dict[(i+1, 0)] = compute_C_i_j(C_i_j_dict[(i-1, 0)], C_i_j_dict[(i, 0)])
    for i in range(1, k-2):
        C_i_j_dict[(i+1, 1)] = compute_C_i_j(C_i_j_dict[(i-1, 1)], C_i_j_dict[(i, 1)])
    return C_i_j_dict

def recover_a(c_prime, c):
    F = c.parent().base()
    n = len(c)
    quotients_list = []
    for i in range(2, n):
        quotients_list.append(c_prime[i]/c[i])
    quotients = vector(F, quotients_list)
    for elt in F.list()[1:]:
        new_quotients = (elt*quotients).list()
        if not 1 in new_quotients:
            return [0, 1] + [1/(1-new_quotients[i]) for i in range(n-2)]

def recover_b(a, k, c):
    n = len(c)
    b_rest = [c[i]/(a[i]^(k-1)) for i in range(1, n)]
    return b_rest

def attack(G):
    F = G.parent().base()
    k = G.nrows()
    n = G.ncols()
    if k > n/2:
        print("attack dual")
        # not yet correct. Gives dual with vectors for dual
        return attack(LinearCode(G).parity_check_matrix())
    C_i_j_dict = compute_all_C_i_j(G)
    c = vector(C_i_j_dict[(k-1, 0)])
    c_prime = vector(C_i_j_dict[(k-2, 1)])
    a = recover_a(c_prime, c)
    #print("recover b")
    b_rest = recover_b(a, k, c)
    print("recover code")
    for elt in F.list()[1:]:
        grs_code = codes.GeneralizedReedSolomonCode(a, k, [elt] + b_rest)
        grs_gen_mat = grs_code.generator_matrix()
        if LinearCode(grs_gen_mat) == LinearCode(G):
            return grs_code, a, [elt] + b_rest